# ทดลองปรับปรุง SVM สำหรับภาพอาหาร

เปรียบเทียบ 3 แนวทางบน `02_denoise_clahe` โดยใช้ split เดิม: (1) HOG + Linear SVM (2) HOG + color histogram + Linear SVM และ (3) pretrained ResNet-50 embeddings + Linear SVM เปรียบเทียบกับ HOG + RBF SVM เดิมในฐานะ baseline แยก pretrained จาก handcrafted features ให้ชัดเจน

เลือกวิธีและ `C` จาก validation macro-F1 เท่านั้น จากนั้นประเมิน test เพียงโมเดลที่เลือก คะแนน train ใช้ดู overfitting ไม่ใช่เกณฑ์เลือกโมเดล ผล test ที่เคยตรวจไปแล้วไม่ถือเป็น blind test อย่างสมบูรณ์

ต้องมีข้อมูลที่ `data/dataset/02_denoise_clahe/{train,val,test}/<class>/` และแพ็กเกจ numpy, pandas, matplotlib, pillow, opencv-python, scikit-image, scikit-learn, torch, torchvision, joblib, jupyter; ResNet-50 ใช้น้ำหนัก ImageNet V2 และอาจดาวน์โหลดครั้งแรก ปรับ `RUN_MODE` เป็น `smoke` เพื่อทดสอบ pipeline ก่อน

## ผลการทดลองเดิมที่กู้จากไฟล์

ตารางและคะแนนด้านล่างสร้างจากไฟล์ `results/svm_experiments/validation_results.csv` และ `test_predictions.csv` ที่บันทึกไว้ก่อนหน้า ไม่ต้องฝึกใหม่ สามารถรันเซลล์นี้ซ้ำเพื่ออ่านไฟล์อีกครั้ง ผลเดิมใน output ของแต่ละเซลล์และกราฟที่ไม่ได้บันทึกแยกไม่สามารถกู้จากไฟล์เหล่านี้ได้ และไฟล์ผลลัพธ์ไม่ได้บันทึกเวอร์ชันโค้ดที่ใช้ทดลอง

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / 'results/svm_experiments/validation_results.csv').is_file()), None)
if project is None:
    raise FileNotFoundError('ไม่พบ results/svm_experiments/validation_results.csv')
saved_results = project / 'results/svm_experiments'
validation = pd.read_csv(saved_results / 'validation_results.csv')
predictions = pd.read_csv(saved_results / 'test_predictions.csv')
print('ผล validation ทุก configuration ที่บันทึกไว้:')
print(validation.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('\nเลือกด้วย validation:', validation.iloc[0]['model'], '| C =', validation.iloc[0]['C'])
print('Test accuracy:', round(accuracy_score(predictions['true'], predictions['predicted']), 4))
print('Test macro-F1:', round(f1_score(predictions['true'], predictions['predicted'], average='macro'), 4))


ผล validation ทุก configuration ที่บันทึกไว้:
                     model          features       C  train_macro_f1  val_macro_f1   seconds
ResNet embeddings + Linear ResNet embeddings  0.0100          1.0000        0.8589   52.6102
ResNet embeddings + Linear ResNet embeddings  0.1000          1.0000        0.8459  645.2811
ResNet embeddings + Linear ResNet embeddings  1.0000          1.0000        0.8429 2578.1947
ResNet embeddings + Linear ResNet embeddings 10.0000          1.0000        0.8146   79.9552
      HOG + color + Linear       HOG + color  0.0100          0.9013        0.4049   43.2939
      HOG + color + Linear       HOG + color  0.1000          0.9943        0.3703  361.1701
      HOG + color + Linear       HOG + color  1.0000          1.0000        0.3474 2898.6573
        HOG + RBF baseline               HOG 10.0000          1.0000        0.3460   57.9034
      HOG + color + Linear       HOG + color 10.0000          1.0000        0.3432 3754.3639
        HOG + RBF baseli

## กราฟจากผลที่บันทึกไว้ — ไม่ต้องเทรนใหม่

รันเฉพาะเซลล์ต่อไปนี้ได้ทันทีหลังเปิด notebook โดยไม่ต้องรันเซลล์ก่อนหน้า เซลล์อ่านผล full run จาก `results/svm_experiments` และสร้างกราฟ validation กับ confusion matrix ของโมเดลที่เลือก

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / 'results/svm_experiments/validation_results.csv').is_file()), None)
if project is None:
    raise FileNotFoundError('ไม่พบ results/svm_experiments/validation_results.csv')
saved_dir = project / 'results/svm_experiments'
validation = pd.read_csv(saved_dir / 'validation_results.csv')
predictions = pd.read_csv(saved_dir / 'test_predictions.csv')

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, group in validation.groupby('model', sort=False):
    group = group.sort_values('C')
    ax.plot(group['C'], group['val_macro_f1'], marker='o', label=name)
ax.set(xscale='log', xlabel='C', ylabel='Validation macro-F1', title='SVM: validation macro-F1 by method and C', ylim=(0, 1))
ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); plt.show()

labels = sorted(set(predictions['true']) | set(predictions['predicted']))
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(predictions['true'], predictions['predicted'], labels=labels,
    display_labels=labels, normalize='true', xticks_rotation=90, ax=ax, colorbar=False, values_format='.2f')
ax.set_title('Chosen SVM: test confusion matrix (row normalized)')
fig.tight_layout(); plt.show()

## 1. ตั้งค่าและตรวจข้อมูล

In [ ]:
import os, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from skimage.feature import hog
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report, ConfusionMatrixDisplay
from IPython.display import display
import joblib

SEED = 42
RUN_MODE = os.environ.get('SVM_RUN_MODE', 'full')  # full | smoke
assert RUN_MODE in ('full', 'smoke')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
HERE = Path.cwd().resolve()
PROJECT = next((p for p in (HERE, *HERE.parents) if (p / 'data/dataset/02_denoise_clahe').is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError('ไม่พบ data/dataset/02_denoise_clahe ในโฟลเดอร์โปรเจกต์')
DATA = PROJECT / 'data/dataset/02_denoise_clahe'
os.environ.setdefault('TORCH_HOME', str(PROJECT / 'checkpoints/torch'))
OUT = PROJECT / 'results' / ('svm_experiments_smoke' if RUN_MODE == 'smoke' else 'svm_experiments')
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('ข้อมูล:', DATA, '| mode:', RUN_MODE, '| device:', DEVICE)

In [ ]:
classes = sorted(p.name for p in (DATA / 'train').iterdir() if p.is_dir())
assert len(classes) >= 2
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
frames = {}
for split in ('train', 'val', 'test'):
    present = sorted(p.name for p in (DATA / split).iterdir() if p.is_dir())
    assert present == classes, f'{split}: class folders ไม่ตรงกับ train'
    rows = []
    for label, name in enumerate(classes):
        files = sorted(p for p in (DATA / split / name).iterdir() if p.suffix.lower() in extensions)
        if RUN_MODE == 'smoke':
            files = files[:3]
        assert files, f'ไม่มีภาพใน {split}/{name}'
        rows.extend((str(p), label, name) for p in files)
    frames[split] = pd.DataFrame(rows, columns=['path', 'y', 'label'])
display(pd.DataFrame({s: f.groupby('label').size() for s, f in frames.items()}))
print('ใช้ split ของทีมตามโฟลเดอร์; ไม่ได้ตรวจ duplicate ข้าม split ในโน้ตบุ๊กนี้')

## 2. สร้าง features
ภาพผ่าน denoise และ CLAHE มาแล้ว ไม่ทำซ้ำ HOG และ histogram ใช้ภาพที่เติมขอบเป็นสี่เหลี่ยมเหมือนต้นฉบับ ส่วน ResNet ใช้ preprocessing มาตรฐานของน้ำหนัก pretrained โดยไม่ fine-tune
Color histogram คำนวณจาก RGB โดยตรง ส่วน HOG ยังใช้ grayscale ตามนิยาม features เดิม

In [ ]:
def pad_to_square(img):
    w, h = img.size
    side = max(w, h)
    result = Image.new('RGB', (side, side), (124, 116, 104))
    result.paste(img, ((side-w)//2, (side-h)//2))
    return result

def handcrafted(path):
    with Image.open(path) as img:
        rgb = np.asarray(pad_to_square(img.convert('RGB')).resize((128, 128), Image.Resampling.BILINEAR))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    edges = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm='L2-Hys').astype(np.float32)
    hist = np.histogramdd(rgb.reshape(-1, 3), bins=(8, 4, 4), range=((0, 256), (0, 256), (0, 256)))[0].astype(np.float32)
    hist /= max(hist.sum(), 1)
    return edges, np.concatenate((edges, hist.ravel()))

weights = ResNet50_Weights.IMAGENET1K_V2
transform = weights.transforms()
embedder = resnet50(weights=weights)
embedder.fc = torch.nn.Identity()
embedder = embedder.to(DEVICE).eval()

@torch.inference_mode()
def features(frame, batch_size=32):
    hogs, colors, embeddings = [], [], []
    for start in range(0, len(frame), batch_size):
        paths = frame.path.iloc[start:start + batch_size]
        batch = []
        for path in paths:
            h, hc = handcrafted(path)
            hogs.append(h); colors.append(hc)
            with Image.open(path) as img:
                batch.append(transform(pad_to_square(img.convert('RGB'))))
        embeddings.append(embedder(torch.stack(batch).to(DEVICE)).cpu().numpy())
    return {'HOG': np.stack(hogs), 'HOG + color': np.stack(colors), 'ResNet embeddings': np.concatenate(embeddings)}

X = {}
for split in ('train', 'val'):  # test is loaded only after model selection
    t = time.time(); X[split] = features(frames[split])
    print(split, {k: v.shape for k, v in X[split].items()}, f'{time.time()-t:.1f}s')
del embedder
if DEVICE.type == 'mps': torch.mps.empty_cache()

## 3. ทดลอง SVM และเลือกด้วย validation
`StandardScaler` เรียนรู้จาก train ภายใน pipeline เท่านั้น Linear SVM ใช้ LinearSVC; baseline ใช้ RBF SVC แบบเดิมและ gamma = 1 / จำนวน HOG features ทดลองค่า C หลายระดับ ไม่มี augmentation เพื่อให้ทุกวิธีใช้ภาพฝึกชุดเดียวกัน

In [ ]:
C_VALUES = [0.01, 0.1, 1, 10] if RUN_MODE == 'full' else [0.1, 1]
experiments = [('HOG + RBF baseline', 'HOG', 'rbf'), ('HOG + Linear', 'HOG', 'linear'),
               ('HOG + color + Linear', 'HOG + color', 'linear'), ('ResNet embeddings + Linear', 'ResNet embeddings', 'linear')]
rows, models = [], {}
y_train, y_val = frames['train'].y.to_numpy(), frames['val'].y.to_numpy()
for name, feature, kernel in experiments:
    for c in C_VALUES:
        estimator = (SVC(C=c, kernel='rbf', gamma=1 / X['train'][feature].shape[1]) if kernel == 'rbf'
                     else LinearSVC(C=c, dual='auto', max_iter=20000, random_state=SEED))
        model = make_pipeline(StandardScaler(), estimator)
        t = time.time(); model.fit(X['train'][feature], y_train)
        train_f1 = f1_score(y_train, model.predict(X['train'][feature]), average='macro', zero_division=0)
        val_f1 = f1_score(y_val, model.predict(X['val'][feature]), average='macro', zero_division=0)
        rows.append({'model': name, 'features': feature, 'C': c, 'train_macro_f1': train_f1, 'val_macro_f1': val_f1, 'seconds': time.time()-t})
        models[(name, c)] = model
        print(f'{name:28s} C={c:<5g} train={train_f1:.3f} val={val_f1:.3f} ({time.time()-t:.1f}s)')
results = pd.DataFrame(rows).sort_values('val_macro_f1', ascending=False, kind='stable').reset_index(drop=True)
display(results.round(4))
best = results.iloc[0]
chosen = models[(best['model'], best['C'])]
print('เลือกจาก validation:', best['model'], '| C =', best['C'])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, group in results.groupby('model', sort=False):
    group = group.sort_values('C')
    ax.plot(group.C, group.val_macro_f1, marker='o', label=name)
ax.set(xscale='log', xlabel='C', ylabel='Validation macro-F1', title='SVM: validation macro-F1 by method and C', ylim=(0, 1))
ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); plt.show()

## 4. ประเมินโมเดลที่เลือกบน test
ผลนี้ใช้ test เพียงครั้งเดียวใน notebook ต่อการรันหนึ่งรอบ อย่าปรับวิธีหรือ C จากผล test แล้วกลับมารายงาน test เดิมว่าเป็นการประเมินอิสระ

In [ ]:
# Extract only the feature family chosen on validation.
test_frame = frames['test']
if best['features'] in ('HOG', 'HOG + color'):
    test_pairs = [handcrafted(p) for p in test_frame.path]
    X_test = np.stack([p[0 if best['features'] == 'HOG' else 1] for p in test_pairs])
else:
    embedder = resnet50(weights=weights); embedder.fc = torch.nn.Identity(); embedder = embedder.to(DEVICE).eval()
    X_test = features(test_frame)['ResNet embeddings']
    del embedder
y_test = test_frame.y.to_numpy()
pred = chosen.predict(X_test)
print('โมเดล:', best['model'], '| C:', best['C'])
print('Test accuracy:', round(accuracy_score(y_test, pred), 4))
print('Test macro-F1:', round(f1_score(y_test, pred, average='macro', zero_division=0), 4))
display(pd.DataFrame(classification_report(y_test, pred, target_names=classes, output_dict=True, zero_division=0)).T.round(3))
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=classes, normalize='true', xticks_rotation=90, ax=ax, colorbar=False, values_format='.2f')
ax.set_title('Chosen SVM: test confusion matrix (row normalized)'); fig.tight_layout(); plt.show()

## 5. บันทึกผล
ไฟล์โมเดลเป็น pipeline ของ scaler และ SVM โดยต้องสร้าง features ด้วยวิธีเดียวกับด้านบนก่อนทำนาย

In [ ]:
results.to_csv(OUT / 'validation_results.csv', index=False)
joblib.dump({'model': chosen, 'feature': best['features'], 'model_name': best['model'], 'C': float(best['C']), 'classes': classes,
             'resnet_weights': 'IMAGENET1K_V2' if best['features'] == 'ResNet embeddings' else None}, OUT / 'chosen_svm.joblib')
pd.DataFrame({'path': test_frame.path, 'true': [classes[i] for i in y_test], 'predicted': [classes[i] for i in pred]}).to_csv(OUT / 'test_predictions.csv', index=False)
print('บันทึกผลที่', OUT)

## 6. ทดลอง freeze น้ำหนัก ResNet-50 แบบชัดเจน

สกัด features ใหม่ด้วย backbone ที่ตั้ง `requires_grad_(False)` แล้วฝึก Linear SVM และเลือก C จาก validation เท่านั้น วิธีนี้ควรให้ features เหมือนข้อ 2 เพราะข้อ 2 ใช้ inference และไม่ได้อัปเดตน้ำหนัก ResNet อยู่แล้ว จึงไม่คาดว่าจะเพิ่มคะแนนเพียงเพราะสั่ง freeze เพิ่ม การตรวจ test ซ้ำไม่ใช่การประเมินแบบ blind

In [ ]:
import hashlib

# ใช้ labels และค่า C จาก split และโหมดปัจจุบัน โดยไม่ต้องรันข้อ 3
C_VALUES = [0.01, 0.1, 1, 10] if RUN_MODE == 'full' else [0.1, 1]
y_train = frames['train'].y.to_numpy()
y_val = frames['val'].y.to_numpy()

def backbone_hash(model):
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

embedder = resnet50(weights=weights)
embedder.fc = torch.nn.Identity()
embedder.requires_grad_(False)
embedder = embedder.to(DEVICE).eval()
assert all(not p.requires_grad for p in embedder.parameters())
before_hash = backbone_hash(embedder)

X_frozen = {}
for split in ('train', 'val'):
    X_frozen[split] = features(frames[split])['ResNet embeddings']
    np.testing.assert_allclose(X_frozen[split], X[split]['ResNet embeddings'], rtol=1e-4, atol=1e-5)
    print(split, 'features ตรงกับข้อ 2:', X_frozen[split].shape)

frozen_rows = []
for c in C_VALUES:
    frozen_svm = make_pipeline(StandardScaler(), LinearSVC(C=c, dual='auto', max_iter=20000, random_state=SEED))
    frozen_svm.fit(X_frozen['train'], y_train)
    frozen_rows.append({'C': c,
        'train_macro_f1': f1_score(y_train, frozen_svm.predict(X_frozen['train']), average='macro', zero_division=0),
        'val_macro_f1': f1_score(y_val, frozen_svm.predict(X_frozen['val']), average='macro', zero_division=0)})

assert backbone_hash(embedder) == before_hash, 'น้ำหนักหรือ buffers ของ ResNet เปลี่ยนระหว่างทดลอง'
frozen_results = pd.DataFrame(frozen_rows).sort_values('val_macro_f1', ascending=False)
display(frozen_results.round(4))
print('ยืนยัน: freeze ทุก parameter และ weights/buffers ไม่เปลี่ยน')
print('เปรียบเทียบกับ ResNet embeddings + Linear ในข้อ 3; ความต่างเล็กน้อยอาจมาจากการคำนวณตัวเลข')
del embedder

## 7. SVM ใช้ค่าพิกเซล RGB โดยตรง

เปรียบเทียบ baseline แบบไม่ใช้ HOG หรือ pretrained backbone: เติมขอบภาพให้เป็นสี่เหลี่ยมเหมือนเดิม แล้ว resize เป็น 32 × 32 ก่อนแปลง RGB เป็นเวกเตอร์ 3,072 ค่า และฝึก Linear SVM เลือก `C` ด้วย validation macro-F1 ก่อนประเมิน test ของค่าที่เลือก เซลล์นี้รันแยกได้ ไม่ต้องรันข้อ 1–6

ภาพขนาด 32 × 32 เป็นการลดต้นทุนคำนวณและอาจทำให้รายละเอียดหาย ข้อมูล test นี้เคยใช้ประเมินมาแล้วจึงไม่ใช่ blind test หากใช้ผลเพื่อตัดสินใจปรับโมเดลต่อ

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score
from IPython.display import display
import joblib

RUN_MODE = os.environ.get('SVM_RUN_MODE', 'full')  # full | smoke
assert RUN_MODE in ('full', 'smoke')
here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / 'data/dataset/02_denoise_clahe').is_dir()), None)
if project is None:
    raise FileNotFoundError('ไม่พบ data/dataset/02_denoise_clahe')
data = project / 'data/dataset/02_denoise_clahe'
classes = sorted(p.name for p in (data / 'train').iterdir() if p.is_dir())
assert len(classes) >= 2
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def load_pixels(split):
    assert sorted(p.name for p in (data / split).iterdir() if p.is_dir()) == classes, f'{split}: class ไม่ตรงกับ train'
    images, labels = [], []
    for label, name in enumerate(classes):
        paths = sorted(p for p in (data / split / name).iterdir() if p.suffix.lower() in extensions)
        if RUN_MODE == 'smoke':
            paths = paths[:3]
        assert paths, f'ไม่มีภาพใน {split}/{name}'
        for path in paths:
            with Image.open(path) as img:
                img = img.convert('RGB')
                w, h = img.size
                canvas = Image.new('RGB', (max(w, h), max(w, h)), (124, 116, 104))
                canvas.paste(img, ((max(w, h) - w) // 2, (max(w, h) - h) // 2))
                images.append(np.asarray(canvas.resize((32, 32), Image.Resampling.BILINEAR), dtype=np.float32).ravel() / 255)
            labels.append(label)
    return np.stack(images), np.array(labels)

X_train_raw, y_train_raw = load_pixels('train')
X_val_raw, y_val_raw = load_pixels('val')
assert X_train_raw.shape[1] == X_val_raw.shape[1] == 32 * 32 * 3
raw_rows, raw_models = [], {}
for c in ([0.01, 0.1, 1, 10] if RUN_MODE == 'full' else [0.1, 1]):
    model = make_pipeline(StandardScaler(), LinearSVC(C=c, dual='auto', max_iter=20000, random_state=42))
    model.fit(X_train_raw, y_train_raw)
    raw_rows.append({'C': c,
        'train_macro_f1': f1_score(y_train_raw, model.predict(X_train_raw), average='macro', zero_division=0),
        'val_macro_f1': f1_score(y_val_raw, model.predict(X_val_raw), average='macro', zero_division=0)})
    raw_models[c] = model
    print(f'C={c} เสร็จแล้ว', flush=True)
raw_results = pd.DataFrame(raw_rows).sort_values('val_macro_f1', ascending=False, kind='stable')
display(raw_results.round(4))
best_c = float(raw_results.iloc[0]['C'])
raw_model = raw_models[best_c]
X_test_raw, y_test_raw = load_pixels('test')
raw_pred = raw_model.predict(X_test_raw)
print(f'เลือก C={best_c} จาก validation | test accuracy={accuracy_score(y_test_raw, raw_pred):.4f} | test macro-F1={f1_score(y_test_raw, raw_pred, average="macro", zero_division=0):.4f}')
out = project / 'results' / ('svm_raw_pixels_smoke' if RUN_MODE == 'smoke' else 'svm_raw_pixels')
out.mkdir(parents=True, exist_ok=True)
raw_results.to_csv(out / 'validation_results.csv', index=False)
joblib.dump({'model': raw_model, 'classes': classes, 'image_size': (32, 32), 'pad_fill': (124, 116, 104)}, out / 'chosen_svm.joblib')
print('บันทึกผลที่', out)